# LoCoMo Eval for Fastrr — Step by Step

Run each cell in order. Uses the same config as Fastrr (Ollama by default).

1. **Setup** — Imports, paths, config
2. **Load dataset** — Load locomo10.json
3. **Create storage** — Repo + Fastrr memory
4. **Step 1: Ingest** — Load conversations into memory
5. **Inspect** — See what was stored
6. **Build QA agents** — Answer generator + grader
7. **Step 2a: One question** — Recall → generate → grade for a single question
8. **Step 2b: All questions** — Run full QA loop
9. **Step 3: Results** — Aggregate and save

## 1. Setup

In [1]:
import os
import sys
import json
import time
from pathlib import Path
from collections import defaultdict

# Paths: detect repo root and add to path so evals + fastrr import
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "locomo_eval.ipynb").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent  # cwd is evals/locomo
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

EVALS_DIR = PROJECT_ROOT / "evals"
DATASETS_DIR = EVALS_DIR / "datasets"
DATASET_PATH = DATASETS_DIR / "locomo10.json"
OUTPUT_DIR = EVALS_DIR / "output"

# Disable Agno telemetry before importing Agno
os.environ.setdefault("AGNO_TELEMETRY", "false")

from agno.agent import Agent
from fastrr import Fastrr
from fastrr.core.config import FastrrConfig
from evals.locomo.ingest import ingest_locomo
from evals.fake_repo import FakeRepoManager

cfg = FastrrConfig()
print(f"Config: provider={cfg.provider} model={cfg.model}")
print(f"Dataset: {DATASET_PATH}")
print(f"Exists: {DATASET_PATH.exists()}")

Config: provider=ollama model=qwen3.5:4b
Dataset: /Users/tarunkhilani/Documents/projects/fastrr/evals/datasets/locomo10.json
Exists: True


## 2. Load dataset

In [2]:
with open(DATASET_PATH) as f:
    data = json.load(f)

num_users = min(10, len(data))
total_qa = sum(
    len([q for q in data[i].get("qa", []) if q.get("category") != 5])
    for i in range(num_users)
)
print(f"Conversations: {num_users}")
print(f"Total QA (excl. abstention): {total_qa}")
print(f"Sample keys per conversation: {list(data[0].keys())}")

Conversations: 10
Total QA (excl. abstention): 1540
Sample keys per conversation: ['qa', 'conversation', 'event_summary', 'observation', 'session_summary', 'sample_id']


## 3. Create storage (FakeRepoManager + Fastrr)

In [7]:
root = OUTPUT_DIR / "storage"
storage_path = root / "repo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
storage_path.mkdir(parents=True, exist_ok=True)

# config = FastrrConfig(provider="ollama", model="qwen3.5:9b")

repo = FakeRepoManager(root)
memory = Fastrr(storage_path=storage_path, repo_manager=repo, config=cfg)

print(f"Storage root: {root}")
print("FakeRepoManager (no Git)")

Storage root: /Users/tarunkhilani/Documents/projects/fastrr/evals/output/storage
FakeRepoManager (no Git)


## 4. Step 1: Ingest

In [8]:
INGEST_BLOCK_SIZE = 4  # number of messages per remember() call

def _format_message(msg: dict, session_date: str) -> str:
    """Format a single message for remember()."""
    speaker = msg.get("speaker", "Unknown")
    text = msg.get("text", "")
    blip = msg.get("blip_caption")
    img_desc = f" (image: {blip})" if blip else ""
    return f"[{session_date}] {speaker}: {text}{img_desc}"

def _ingest_session(memory, session: list, session_date: str, block_size: int = INGEST_BLOCK_SIZE):
    """Ingest a session's messages in blocks of block_size."""
    from tqdm.notebook import tqdm
    formatted = [_format_message(msg, session_date) for msg in session]
    total_blocks = (len(formatted) + block_size - 1) // block_size
    for i in tqdm(range(0, len(formatted), block_size), total=total_blocks, desc="Ingesting blocks"):
        block = formatted[i : i + block_size]
        content = "\n".join(block)
        print(content)
        print()
        memory.remember(content)

path = Path(DATASET_PATH)
with open(path) as f:
    data = json.load(f)

for group_idx in range(min(num_users, len(data))):
    sample = data[group_idx]
    conversation = sample.get("conversation", {})

    for session_idx in range(35):
        session_key = f"session_{session_idx}"
        session = conversation.get(session_key)
        if session is None:
            continue

        date_key = f"session_{session_idx}_date_time"
        session_date = conversation.get(date_key, "unknown")

        _ingest_session(memory, session, session_date)
        break
    break


Ingesting blocks:   0%|          | 0/5 [00:00<?, ?it/s]

[1:56 pm on 8 May, 2023] Caroline: Hey Mel! Good to see you! How have you been?
[1:56 pm on 8 May, 2023] Melanie: Hey Caroline! Good to see you! I'm swamped with the kids & work. What's up with you? Anything new?
[1:56 pm on 8 May, 2023] Caroline: I went to a LGBTQ support group yesterday and it was so powerful.
[1:56 pm on 8 May, 2023] Melanie: Wow, that's cool, Caroline! What happened that was so awesome? Did you hear any inspiring stories?

[1:56 pm on 8 May, 2023] Caroline: The transgender stories were so inspiring! I was so happy and thankful for all the support. (image: a photo of a dog walking past a wall with a painting of a woman)
[1:56 pm on 8 May, 2023] Melanie: Wow, love that painting! So cool you found such a helpful group. What's it done for you?
[1:56 pm on 8 May, 2023] Caroline: The support group has made me feel accepted and given me courage to embrace myself.
[1:56 pm on 8 May, 2023] Melanie: That's really cool. You've got guts. What now?

[1:56 pm on 8 May, 2023] Car

In [ ]:
# start = time.monotonic()
# ingest_locomo(memory, DATASET_PATH, num_users=num_users, log=print)
# elapsed = time.monotonic() - start
# print(f"\nDone. Ingested {num_users} conversations in {elapsed:.1f}s")

## 5. Inspect what was stored

In [ ]:
history = memory.history()
print(f"{history=}")

history=[]


## 6. Build QA agents (answer generator + grader)

In [11]:
if cfg.provider == "openrouter":
    from agno.models.openrouter import OpenRouter
    eval_model = OpenRouter(id=cfg.model, api_key=cfg.openrouter_api_key)
else:
    from agno.models.ollama import Ollama
    eval_model = Ollama(id=cfg.model, host=cfg.ollama_host)

answer_agent = Agent(
    model=eval_model,
    instructions="Answer concisely based only on the context provided.",
)
grade_agent = Agent(
    model=eval_model,
    instructions="Return exactly CORRECT or WRONG as your final answer.",
)
print("Agents ready.")

Agents ready.


## 7. Step 2a: Run one question (recall → generate → grade)

In [13]:
ANSWER_PROMPT = """You are a helpful expert answering questions based on the provided context.
Context:
{context}

Question: {question}
Answer:"""

GRADE_PROMPT = """Label the answer as CORRECT or WRONG.
Question: {question}
Gold: {gold}
Generated: {response}
Return CORRECT or WRONG."""

qa = data[0]["qa"][0]
question = qa["question"]
gold = qa["answer"]

print("Question:", question)
print("Gold:", gold)
print()

print("Recalling...")
context = memory.recall(query=question)
print(f"Context length: {len(context)} chars")
print(f"{context=}")
print()

print("Generating answer...")
prompt = ANSWER_PROMPT.format(context=context, question=question)
answer_result = answer_agent.run(prompt)
answer = (answer_result.content or "").strip()
print("Answer:", answer)
print()

print("Grading...")
grade_prompt = GRADE_PROMPT.format(question=question, gold=gold, response=answer)
grade_result = grade_agent.run(grade_prompt)
text = (grade_result.content or "").strip().upper()
grade = "CORRECT" in text and "WRONG" not in text and "INCORRECT" not in text
print(f"Grade: {"CORRECT" if grade else "WRONG"}")

Question: When did Caroline go to the LGBTQ support group?
Gold: 7 May 2023

Recalling...
Context length: 141 chars

Generating answer...
Answer: May 7, 2023.

Grading...
Grade: CORRECT


## 8. Step 2b: Run all questions

In [ ]:
CATEGORY_NAMES = {1: "single_hop", 2: "temporal", 3: "multi_hop", 4: "open_domain", 5: "abstention"}

def generate_answer(agent, context, question):
    p = ANSWER_PROMPT.format(context=context, question=question)
    r = agent.run(p)
    return (r.content or "").strip()

def grade_answer(agent, question, gold, response):
    p = GRADE_PROMPT.format(question=question, gold=str(gold), response=response)
    r = agent.run(p)
    t = (r.content or "").strip().upper()
    return "CORRECT" in t and "WRONG" not in t and "INCORRECT" not in t

results = defaultdict(list)
scores_by_cat = defaultdict(list)
step_start = time.monotonic()
done = 0

for group_idx in range(num_users):
    user_id = f"locomo_{group_idx}"
    qa_filtered = [q for q in data[group_idx].get("qa", []) if q.get("category") != 5]
    for qa in qa_filtered:
        question = qa.get("question", "")
        gold = qa.get("answer")
        if gold is None:
            continue
        context = memory.recall(user_id, query=question)
        answer = generate_answer(answer_agent, context, question)
        grade = grade_answer(grade_agent, question, gold, answer)
        done += 1
        cat_name = CATEGORY_NAMES.get(qa.get("category", 0), "unknown")
        scores_by_cat[cat_name].append(grade)
        results[user_id].append({"question": question, "answer": answer, "golden_answer": gold, "grade": grade, "category": cat_name})
        if done % 50 == 0 or done == total_qa:
            print(f"Progress: {done}/{total_qa} | {time.monotonic() - step_start:.0f}s")

print(f"\nQA done in {time.monotonic() - step_start:.1f}s")

## 9. Step 3: Aggregate and save results

In [ ]:
total = sum(len(v) for v in results.values())
correct = sum(1 for items in results.values() for it in items if it["grade"])
overall = correct / total if total else 0

by_cat = {
    name: sum(s) / len(s) if s else 0
    for name, s in scores_by_cat.items()
    if name != "abstention"
}

print("=== LoCoMo Eval Results ===")
print(f"Overall: {overall:.2%} ({correct}/{total})")
for name, acc in by_cat.items():
    print(f"  {name}: {acc:.2%}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_file = OUTPUT_DIR / "locomo_results.json"
with open(out_file, "w") as f:
    json.dump({
        "overall_accuracy": overall,
        "total_questions": total,
        "correct": correct,
        "by_category": by_cat,
        "results": dict(results),
    }, f, indent=2)
print(f"\nResults saved to {out_file}")